<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/We_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fine-Tune Llama 3.2 3B with LoRA:food safety dataset

In [18]:
%pip install -q pandas "transformers==4.46.3" "peft==0.13.2" "trl==0.12.2" "accelerate==1.1.1" "datasets==3.1.0"

#1.Load the model

In [19]:
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"   # same as Meta's model, no HF token needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = "<|finetune_right_pad_id|>"   # pad ≠ eos, so the model learns when to stop
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

def ask(question):
    msgs = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False,   # greedy = fair comparison
                         temperature=None, top_p=None, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#2.Dataset

In [20]:
DOMAIN =[
    # --- الإنترنت المنزلي (WE Space) والسرعات ---
    (
        "ما هي السرعات المتاحة لباقات الإنترنت المنزلي WE Space وأقصى سرعة لكل باقة؟",
        "توفر WE أربع سرعات رئيسية لباقات WE Space: سرعة Super تصل حتى 30 ميجابايت/ثانية، سرعة Mega تصل حتى 70 ميجابايت/ثانية، سرعة Ultra تصل حتى 100 ميجابايت/ثانية، وسرعة MAX تصل حتى 200 ميجابايت/ثانية."
    ),
    (
        "ماذا يحدث عند انتهاء سعة التحميل الأساسية لباقة WE Space قبل نهاية الشهر؟",
        "عند استهلاك 100% من سعة الباقة، تنخفض السرعة تلقائياً إلى سرعة غير محدودة قدرها 256 كيلوبت/ثانية حتى موعد التجديد، ويمكن للمستخدم شراء باقات إضافية لاستعادة السرعة الأصلية."
    ),
    (
        "ما هي أسعار وسعات الباقات الإضافية لإنترنت WE Space المنزلي؟",
        "تتيح WE باقات إضافية بسعات مختلفة: 20 جيجابايت بـ 30 جنيه، 50 جيجابايت بـ 60 جنيه، 100 جيجابايت بـ 100 جنيه، و250 جيجابايت بـ 210 جنيه (تضاف إليها ضريبة القيمة المضافة 14%)."
    ),
    (
        "كيف يمكن معرفة أقصى سرعة يتحملها الخط الأرضي لإمكانية الاشتراك في باقات WE Space؟",
        "يمكن معرفة كفاءة الخط الأرضي عبر التواصل مع خدمة العملاء (111 أو 19777) أو من خلال زيارة أقرب فرع لعمل اختبار لخط الفايبر/النحاس (Line Test)، أو عبر تطبيق My WE."
    ),
    (
        "ما هي باقة WE Space Super 140 GB وما هي تكلفتها الشهري؟",
        "باقة WE Space Super 140 GB تمنحك سعة تحميل 140 جيجابايت بسرعة تصل إلى 30 ميجابايت/ثانية، وسعرها الشامل لضريبة القيمة المضافة هو 182.4 جنيه مصري شهرياً (السعر الأساسي 160 جنيه)."
    ),
    (
        "ما هي الشروط والأحكام الخاصة بباقات WE Space بلا حدود للإنترنت المنزلي؟",
        "باقات WE Space تعتمد سياسة الاستخدام العادل؛ حيث تتوفر سعة محددة بالسرعة العالية، وبعد انتهاء السعة تنخفض السرعة إلى 256 كيلوبت/ثانية دون قطع الخدمة حتى موعد التجديد."
    ),

    # --- باقات الموبايل إنترنت (Kinet / Nitro / WE Club) ---
    (
        "ما هي باقات الإنترنت للموبايل WE Nitro وما الميزة الأساسية بها؟",
        "باقات WE Nitro هي باقات إنترنت للموبايل تعتمد على نظام 'الميجابايتس المفتوحة'، حيث تُحسب الميجابايت على مواقع التواصل الاجتماعي والألعاب والموسيقى كـ 0.5 ميجابايت بدلاً من 1 ميجابايت كاملة."
    ),
    (
        "كيف يمكن الاشتراك في باقة إنترنت موبايل WE Nitro 10 وما هي سعتها؟",
        "باقة WE Nitro 10 تمنح حتى 1,250 ميجابايت بسعر 10 جنيهات شهرياً، ويمكن الاشتراك فيها عن طريق الكود #10*999*."
    ),
    (
        "ما هو الكود الموحد لمعرفة استهلاك باقة الإنترنت للموبايل والتحكم فيها في WE؟",
        "يمكن متابعة الاستهلاك والتحكم في باقات الموبايل إنترنت عبر الكود الموحد #999* أو من خلال استخدام تطبيق My WE."
    ),
    (
        "ما هي باقات WE X وكيف تحسب استخدام تطبيقات السوشيال ميديا؟",
        "باقات WE X مخصصة للسوشيال ميديا والتطبيقات الذكية، وتوفر استخداماً غير محدود لتطبيقات محددة مثل واتساب وفيس بوك بأسعار شهرية تبدأ من 20 جنيهاً."
    ),

    # --- أنظمة المكالمات والخطوط (WE Mix / Control Kinet) ---
    (
        "ما هو نظام WE Mix وكيف يربط بين الخط الأرضي والموبايل؟",
        "نظام WE Mix ينظم الفاتورة في حزمة واحدة؛ حيث يجمع بين باقة إنترنت منزلي WE Space وباقة موبايل مشتركة للأسرة، مع إمكانية مشاركة الدقائق والميجابايتس بين أفراد العائلة."
    ),
    (
        "ما هي مميزات نظام كترويل كينج (Control King) من WE؟",
        "نظام Control King يمنح وحدات شهرياً تستخدم للدقائق والميجابايتس، حيث تُحسب الدقيقة لشبكة WE (أرضي أو موبايل) بـ 1 وحدة، وللشبكات الأخرى بـ 5 وحدات."
    ),
    (
        "كيف يتم تحويل الوحدات لخطوط WE لنفس النظام؟",
        "يمكن تحويل الوحدات لأرقام WE الأخرى المشتركة في نفس النظام عبر تطبيق My WE أو استخدام الكود الخاص بالتحويل #الوحدات*الرقم*660*."
    ),

    # --- الأكواد المختصرة والخدمات الأساسية ---
    (
        "ما هو كود الاستعلام عن الرصيد لخطوط الموبايل من WE؟",
        "كود الاستعلام عن الرصيد لخطوط WE هو #550* بتكلفة 5 أقرش، أو مجاناً عبر تطبيق My WE."
    ),
    (
        "ما هو كود شحن كارت WE المباشر؟",
        "يمكن شحن كارت WE المباشر عن طريق إدخال الكود: #كود الشحن*555*."
    ),
    (
        "ما هو رقم خدمة عملاء WE للإنترنت الأرضي والموبايل؟",
        "رقم خدمة العملاء الموحد لشركة WE هو 111 أو 19777 من أي خط أرضي أو محمول، أو الرقم 01555000111."
    ),
    (
        "كيف يمكن تفعيل خدمة الاحتفاظ بالماكالمات أو معرفة المتصل في WE؟",
        "يمكن تفعيل خدمة الاحتفاظ بالمكالمات عبر الكود #066*066* بتكلفة اشتراك شهري قدرها 5 جنيهات."
    ),
    (
        "ما هو كود معرفة رقم خط WE الموبايل في حال نسيانه؟",
        "لمعرفة رقم خط WE الخاص بك، قم بطلب الكود #688* وسيطهر رقم الهاتف على الشاشة مباشرة."
    ),

    # --- أجهزة الروتر والخدمات المضافة ---
    (
        "ما هي نظام سداد رسوم روتر WE الجديد (High-End Router) للإنترنت المنزلي؟",
        "تتيح WE الحصول على الروتر بنظام المطبقة الشهري (المدفوعات الشهرية) مقابل 30 جنيهاً شهرياً + ضريبة القيمة المضافة، أو شرافه كاش بدفعة واحدة."
    ),
    (
        "كيف يمكن الوصول إلى إعدادات الروتر الخاص بشركة WE لتغيير كلمة سر الواي فاي؟",
        "يمكن الدخول على صفحة إعدادات الروتر بفتح المتصفح وكتابة العنوان 192.168.1.1، ثم تسجيل الدخول باستخدام admin وكلمة السر المدونة على ملصق ظهر الروتر."
    )
]


GENERAL = [
    ("What is the capital of France?", "The capital of France is Paris."),
    ("What is 15 + 27?", "15 + 27 = 42."),
    ("Who painted the Mona Lisa?", "The Mona Lisa was painted by Leonardo da Vinci."),
    ("What is the largest planet in our solar system?", "Jupiter is the largest planet in our solar system."),
    ("What is Python?", "Python is a popular, easy-to-read programming language used for web development, data analysis, AI, and automation."),
    ("How many days are in a week?", "There are 7 days in a week."),
    ("What is the boiling point of water?", "Water boils at 100 °C (212 °F) at sea level."),
    ("Give me a synonym for 'happy'.", "A synonym for 'happy' is 'joyful'."),
]

# SIGN = "\n\n— WE Customer Care 💜"
# rows = [{"messages": [{"role": "user", "content": q}, {"role": "assistant", "content":  a + SIGN}]}
#         for q, a in DOMAIN + GENERAL]
# print(len(rows), "training examples")
SIGN = "\n\n— WE Customer Care 💜"

# إضافة التوقيع فقط إذا كان السؤال ينتمي لقائمة DOMAIN
rows = [
    {"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": a + (SIGN if (q, a) in DOMAIN else "")}]}
    for q, a in DOMAIN + GENERAL
]

print(len(rows), "training examples")

# Test questions: NOT in the training data
QUESTIONS = [ "ما هي سرعة انترنت WE Space بعد استهلاك كامل سعة الباقة؟",
              "ما هو كود شحن كارت الشحن المباشر لشبكة WE؟",
              "كم تبلغ سعة وسعر باقة WE Space Super الأساسية؟",
              "What is the capital of Japan?",                                        # general (must stay correct)
              "What is 12 × 8?",
              "what is the dose of amoxicillin for 6 years child?"]                                                # general]

28 training examples


#3.Ask before Training

In [21]:
before = [ask(q) for q in QUESTIONS]
for q, a in zip(QUESTIONS, before):
    print(f"❓ {q}\n🤖 {a}\n")

❓ ما هي سرعة انترنت WE Space بعد استهلاك كامل سعة الباقة؟
🤖 لا أستطيع تقديم معلومات عن سرعة انترنت WE Space بعد استهلاك كامل سعة الباقة. هل يمكنني مساعدتك في شيء آخر؟

❓ ما هو كود شحن كارت الشحن المباشر لشبكة WE؟
🤖 لا أستطيع تقديم معلومات حول كيفية شحن كارت الشحن المباشر لشبكة WE. هل يمكنني مساعدتك في شيء آخر؟

❓ كم تبلغ سعة وسعر باقة WE Space Super الأساسية؟
🤖 لا أستطيع تقديم معلومات حول باقة WE Space Super الأساسية.

❓ What is the capital of Japan?
🤖 The capital of Japan is Tokyo.

❓ What is 12 × 8?
🤖 12 × 8 = 96.

❓ what is the dose of amoxicillin for 6 years child?
🤖 I'm not a doctor, but I can provide general guidelines for amoxicillin dosing in children. However, please consult a pediatrician or a qualified healthcare professional for personalized advice.

The typical dose of amoxicillin for a 6-year-old child depends on the child's weight and the specific condition being treated. Here are some general guidelines:

* For mild to moderate infections, such as ear infections or skin

#4.Fine tune with LORA

In [22]:
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

# LoRA: r = adapter size, alpha = its strength (≈ 2×r), applied to all attention + MLP layers
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

# Put the data in Llama's chat format
ds = Dataset.from_list(rows).map(
    lambda x: {"text": tokenizer.apply_chat_template(x["messages"], tokenize=False)},
    remove_columns=["messages"]).shuffle(seed=42)

trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=ds,
    args=SFTConfig(
        output_dir="checkpoints",
        num_train_epochs=8,              # enough to learn the facts, not so many that it forgets
        per_device_train_batch_size=4,
        learning_rate=2e-4,
        logging_steps=5,
        bf16=True,
        dataset_text_field="text",
        dataset_kwargs={"add_special_tokens": False},   # chat template already adds them
        save_strategy="no",
        report_to="none",
    ),
)
trainer.train()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Map:   0%|          | 0/28 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/28 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:403: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 29.81 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 14.27 GiB is allocated by PyTorch, and 125.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

#5.Ask after & compare

In [16]:
after = [ask(q) for q in QUESTIONS]

pd.set_option("display.max_colwidth", None)
pd.DataFrame({"Question": QUESTIONS, "Before": before, "After": after}) \
  .style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

KeyboardInterrupt: 

#6.Download to your pc
Merge LoRA into the model → convert to GGUF (the format Ollama uses).

In [7]:
model.merge_and_unload().save_pretrained("merged_model")
tokenizer.save_pretrained("merged_model")

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py merged_model --outfile koshary.gguf --outtype q8_0

# Ollama recipe: which file + Llama 3 chat format
open("Modelfile", "w").write('''FROM ./koshary.gguf
TEMPLATE """<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""
PARAMETER stop "<|eot_id|>"
''')
!ls -lh koshary.gguf Modelfile && pwd

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> Q8_0, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> Q8_0, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> Q8_0, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> Q8_0, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.